# AI-powered Supplier Risk & Sustainability ScoreCard

## Overview

This system combines **four data artefacts** to produce holistic supplier intelligence:

| Data Source | Purpose | Artefact | 
| --- | --- | --- | 
| Financial Risk Data (D&B) | Bankruptcy probability, credit limits, payment behavior, compliance flags | cnr_catalog.default.`credit_risk_duns` | 
| Supplier-to-DUNS Mapping | Bridges internal supplier IDs to D&B DUNS numbers across all datasets | cnr_catalog.default.`business_partner`| 
| Carbon Emissions Data | Energy use, transport, waste — Scope 3 emissions calculation | cnr_catalog.default.`suppliers_carbon_emission` | 
| Supplier Invoices, Purchase Order, Supplier Data Products from Cloud ERP Intelligence Private | Supplier & Procurement Data Combined together | cnr_catalog.default.`supplier_inv` | 
 

## Capabilities
- **Predict** supplier failure/disruption risk using Gradient Boosting ML
- **Identify** high carbon footprint suppliers impacting ESG goals
- **Recommend** actions: Continue / Monitor / Improve / Replace

## Supports
- Procurement decision-making
- ESG reporting (Scope 3 emissions)
- Risk management & supply chain resilience

### Data Prepation 
Let us begin by preparing the data. To simplify the workshop experience and ensure smooth execution, the SAP-managed data products have already been prepared and persisted in advance. Since hundreds of participants may access the same data products simultaneously, running the joins during the hands-on exercises could lead to performance bottlenecks.

For this reason, the following query has already been executed:

`SELECT 
  s.* EXCEPT(__OPERATION_TYPE, __TIMESTAMP, load_type_8995a2862a8343bd8390aaa82c46e881, run_id_8995a2862a8343bd8390aaa82c46e881),
  si.* EXCEPT(__OPERATION_TYPE, __TIMESTAMP, load_type_8995a2862a8343bd8390aaa82c46e881, run_id_8995a2862a8343bd8390aaa82c46e881),
  sii.* EXCEPT(
    SupplierInvoice, FiscalYear, CompanyCode, DocumentDate, PostingDate,
    SupplierInvoiceIDByInvcgParty, InvoicingParty, IsInvoice, DocumentCurrency,
    SuplrInvcAutomReducedAmount, UnplannedDeliveryCost, DocumentHeaderText,
    SupplierInvoiceOrigin, UnplannedDeliveryCostTaxCode, SupplierInvoiceStatus,
    ReverseDocument, ReverseDocumentFiscalYear,
    __OPERATION_TYPE, __TIMESTAMP, load_type_8995a2862a8343bd8390aaa82c46e881, run_id_8995a2862a8343bd8390aaa82c46e881
  ),
  po.* EXCEPT(
    Supplier, CompanyCode, CreatedByUser, CreationDate, Customer,
    DocumentCurrency, InvoicingParty, PurchaseOrder,
    __OPERATION_TYPE, __TIMESTAMP, load_type_8995a2862a8343bd8390aaa82c46e881, run_id_8995a2862a8343bd8390aaa82c46e881
  )
FROM bdc_supinv.supplierinvoice.supplierinvoice si
INNER JOIN bdc_supinv.supplierinvoice.supplierinvoiceitem sii 
  ON sii.SupplierInvoice = si.SupplierInvoice
INNER JOIN bdc_purchord.purchaseorder.purchaseorder po
  ON po.PurchaseOrder = sii.PurchaseOrder
INNER JOIN `supplier-data-product`.supplier.supplier s
  ON s.Supplier = po.Supplier`

The results of this query have been persisted in the table cnr_catalog.default.supplier_inv, which will serve as the placeholder for the SAP BDC data product throughout this workshop.

The query combines data from the Supplier, Supplier Invoice, and Purchase Order data products shared through the SAP BDC Catalog.

> **Note:** You <b> DO NOT </b> need to run the query above, as it has already been executed for you. The resulting dataset has been persisted in the table `cnr_catalog.default.supplier_inv`, which will be used throughout the workshop as the placeholder for the SAP BDC data product.

We will now load all the data sources into Data frames for further processing. We have to load to load the following data:
- duns_reference_table_enriched_with_risk_data: Shared as a custom data product from BW, this data contains bankruptcy probability, credit limits, payment behavior, compliance flags.
- supplier_duns: Shared as a custom data product from BW, this data is the BusinessPartner information
- suppliers_carbon_emission: Carbon emission information shared with SAP Databricks from a non-SAP system


In [0]:
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Load Financial Risk Data
risk_df = spark.read.table("cnr_catalog.default.credit_risk_duns").toPandas()
print(f"Financial Risk Data: {risk_df.shape}")

# Load Supplier-to-DUNS Mapping Table
supplier_duns_df = spark.read.table("cnr_catalog.default.business_partner").toPandas()
print(f"Supplier-DUNS Mapping: {supplier_duns_df.shape}")

# Load Carbon Emissions Data
carbon_df = spark.read.table("cnr_catalog.default.suppliers_carbon_emission").toPandas()
print(f"Carbon Emissions Data: {carbon_df.shape}")

# Load Supplier Invoices - filtered to suppliers in the mapping table, limited to 1000 rows
invoices_df = spark.sql("""
    SELECT *
    FROM cnr_catalog.default.supplier_inv
    WHERE Supplier IN (SELECT Supplier FROM cnr_catalog.default.business_partner)
    LIMIT 1880
""").toPandas()
print(f"Supplier Invoices Data: {invoices_df.shape}")

print(f"\nSupplier-DUNS mapping sample:")
print(supplier_duns_df[['Supplier', 'DunsNumber', 'SupplierName']].head())
print(f"\nMapping covers {supplier_duns_df['Supplier'].nunique()} unique suppliers to {supplier_duns_df['DunsNumber'].nunique()} unique DUNS numbers")

### Data Integration & Feature Engineering

We aggregate invoice-level data to supplier-level metrics, then join all three datasets using supplier identifiers as the common key.

In [0]:
# --- Aggregate Invoice Data at Supplier Level ---

invoices_df['SupplierInvoiceItemAmount'] = pd.to_numeric(invoices_df['SupplierInvoiceItemAmount'], errors='coerce')

# Aggregate at supplier level
invoice_agg = invoices_df.groupby('Supplier').agg(
    total_spend=('SupplierInvoiceItemAmount', 'sum'),
    num_invoices=('SupplierInvoice', 'count'),
    avg_order_value=('SupplierInvoiceItemAmount', 'mean'),
    num_unique_items=('PurchaseOrderItemMaterial', 'nunique')
).reset_index()

print(f"Invoice aggregation: {invoice_agg.shape[0]} unique suppliers")

# --- Use supplier_duns mapping to join with financial risk data ---
# The mapping table bridges: Supplier ID → DunsNumber
# Carbon data uses 'Supplier Name' = supplier_duns 'Supplier'
# Invoice data uses 'Supplier' = supplier_duns 'Supplier'
# Risk data uses 'DunsNumber' = supplier_duns 'DunsNumber'

# Prepare the mapping table
supplier_duns_df['Supplier'] = supplier_duns_df['Supplier'].astype(str).str.strip()
supplier_duns_df['DunsNumber'] = supplier_duns_df['DunsNumber'].astype(str).str.strip()

# Check key overlap
carbon_keys = set(carbon_df['Supplier Name'].astype(str).str.strip())
invoice_keys = set(invoice_agg['Supplier'].astype(str).str.strip())
mapping_keys = set(supplier_duns_df['Supplier'])
duns_keys = set(supplier_duns_df['DunsNumber'])
risk_duns_keys = set(risk_df['DunsNumber'].astype(str).str.strip())

print(f"\nJoin key overlap analysis (via supplier_duns mapping):")
print(f"  Carbon suppliers in mapping: {len(carbon_keys & mapping_keys)} / {len(carbon_keys)}")
print(f"  Invoice suppliers in mapping: {len(invoice_keys & mapping_keys)} / {len(invoice_keys)}")
print(f"  Mapping DUNS in risk data: {len(duns_keys & risk_duns_keys)} / {len(duns_keys)}")

# Step 1: Start with carbon data, add the DUNS number via mapping
carbon_df['supplier_key'] = carbon_df['Supplier Name'].astype(str).str.strip()
merged_df = carbon_df.merge(
    supplier_duns_df[['Supplier', 'DunsNumber', 'SupplierName']].rename(
        columns={'Supplier': 'supplier_key', 'SupplierName': 'MappedSupplierName'}
    ),
    on='supplier_key',
    how='left'
)

# Step 2: Join invoice aggregates using the supplier key
invoice_agg['supplier_key'] = invoice_agg['Supplier'].astype(str).str.strip()
merged_df = merged_df.merge(invoice_agg, on='supplier_key', how='left')

# Step 3: Join financial risk data using DunsNumber from the mapping
risk_df['DunsNumber'] = risk_df['DunsNumber'].astype(str).str.strip()
merged_df = merged_df.merge(risk_df, on='DunsNumber', how='left')

print(f"\nMerged dataset shape: {merged_df.shape}")
print(f"Suppliers with DUNS mapping: {merged_df['DunsNumber'].notna().sum()}")
print(f"Suppliers with invoice data: {merged_df['total_spend'].notna().sum()}")
print(f"Suppliers with financial risk data: {merged_df['Financial_Stress_Score'].notna().sum()}")

# Show null statistics for key columns
key_cols = ['DunsNumber', 'Financial_Stress_Score', 'Bankruptcy_Probability_Pct',
            'total_spend', 'num_invoices', 'Viability_Score', 'Paydex_Score']
null_pct = (merged_df[key_cols].isnull().sum() / len(merged_df) * 100)
print(f"\nNull % in key columns:")
print(null_pct.round(1))

### Carbon Emissions Calculation

This is the first assessment, where sustainability scores are calculated for each supplier. Standardized emission factors are used to estimate total CO₂ emissions, measure carbon intensity per unit produced, and assign a carbon rating (A, B, or C) to each supplier.

In [0]:
# --- Emission Factors ---
# Electricity: 0.4 kg CO2/kWh (adjusted by renewable %)
# Natural Gas: 0.2 kg CO2/kWh
# Truck: 0.1 kg CO2/km
# Ship: 0.02 kg CO2/km
# Landfill: 0.5 kg CO2/kg
# Recycling: 0.05 kg CO2/kg

merged_df['Electricity_Emissions'] = merged_df['Electricity (kWh)'] * 0.4 * (1 - merged_df['Renewable Energy (%)'] / 100)
merged_df['Gas_Emissions'] = merged_df['Natural Gas (kWh)'] * 0.2
merged_df['Truck_Emissions'] = merged_df['Truck Distance (km)'] * 0.1
merged_df['Ship_Emissions'] = merged_df['Ship Distance (km)'] * 0.02
merged_df['Landfill_Emissions'] = merged_df['Landfill Waste (kg)'] * 0.5
merged_df['Recycling_Emissions'] = merged_df['Recycled Waste (kg)'] * 0.05

# Total Emissions
merged_df['Total_Emissions'] = (
    merged_df['Electricity_Emissions'] +
    merged_df['Gas_Emissions'] +
    merged_df['Truck_Emissions'] +
    merged_df['Ship_Emissions'] +
    merged_df['Landfill_Emissions'] +
    merged_df['Recycling_Emissions']
)

# Carbon Intensity
merged_df['Carbon_Intensity'] = merged_df['Total_Emissions'] / merged_df['Annual Production']

# Carbon Score
def assign_carbon_score(intensity):
    if pd.isna(intensity):
        return 'B'  # default for missing
    elif intensity < 1.5:
        return 'A'
    elif intensity <= 3:
        return 'B'
    else:
        return 'C'

merged_df['Carbon_Score'] = merged_df['Carbon_Intensity'].apply(assign_carbon_score)

print("Carbon Emissions Summary:")
print(f"  Total Scope 3 Emissions: {merged_df['Total_Emissions'].sum():,.0f} kg CO2")
print(f"  Avg Carbon Intensity: {merged_df['Carbon_Intensity'].mean():.2f} kg CO2/unit")
print(f"\nCarbon Score Distribution:")
print(merged_df['Carbon_Score'].value_counts().sort_index())

### Composite Risk Scoring

Based on the emission scores calulcated in the step above and the credit risk information, we will build a multi-dimensional risk score combining:
- **Financial Risk** (40%): stress score, bankruptcy probability, payment behavior
- **Sustainability Risk** (30%): carbon intensity, renewable energy adoption, waste management
- **Operational Risk** (30%): business continuity, legal proceedings, dependencies

In [0]:
from sklearn.preprocessing import MinMaxScaler

# --- Helper: Normalize column to 0-100 scale ---
def normalize_0_100(series):
    """Normalize a series to 0-100 scale, handling NaNs."""
    s = pd.to_numeric(series, errors='coerce')
    min_val = s.min()
    max_val = s.max()
    if max_val == min_val:
        return s.fillna(50).clip(0, 100)
    return ((s - min_val) / (max_val - min_val) * 100).fillna(50)

# --- Financial Risk Score (0-100) ---
merged_df['Financial_Stress_Score_num'] = pd.to_numeric(merged_df.get('Financial_Stress_Score'), errors='coerce')
merged_df['Bankruptcy_Probability_Pct_num'] = pd.to_numeric(merged_df.get('Bankruptcy_Probability_Pct'), errors='coerce')
merged_df['Late_Payment_Pct_num'] = pd.to_numeric(merged_df.get('Late_Payment_Pct'), errors='coerce')
merged_df['Viability_Score_num'] = pd.to_numeric(merged_df.get('Viability_Score'), errors='coerce')
merged_df['Paydex_Score_num'] = pd.to_numeric(merged_df.get('Paydex_Score'), errors='coerce')

financial_stress_norm = normalize_0_100(merged_df['Financial_Stress_Score_num'])
bankruptcy_norm = normalize_0_100(merged_df['Bankruptcy_Probability_Pct_num'])
paydex_inv_norm = 100 - normalize_0_100(merged_df['Paydex_Score_num'])
viability_inv_norm = 100 - normalize_0_100(merged_df['Viability_Score_num'])

merged_df['Financial_Risk_Score'] = (
    financial_stress_norm * 0.25 +
    bankruptcy_norm * 0.30 +
    paydex_inv_norm * 0.20 +
    viability_inv_norm * 0.10
)

# --- Sustainability Risk Score (0-100) ---
carbon_intensity_norm = normalize_0_100(merged_df['Carbon_Intensity'])
renewable_inv_norm = 100 - normalize_0_100(merged_df['Renewable Energy (%)'])

merged_df['waste_ratio'] = merged_df['Landfill Waste (kg)'] / (
    merged_df['Landfill Waste (kg)'] + merged_df['Recycled Waste (kg)'] + 1e-6
)
waste_ratio_norm = normalize_0_100(merged_df['waste_ratio'])

merged_df['Sustainability_Risk_Score'] = (
    carbon_intensity_norm * 0.50 +
    renewable_inv_norm * 0.30 +
    waste_ratio_norm * 0.20
)

# --- Operational Risk Score (0-100) ---
merged_df['Business_Continuity_Score_num'] = pd.to_numeric(merged_df.get('Business_Continuity_Score'), errors='coerce')
merged_df['Active_Legal_Proceedings_num'] = pd.to_numeric(merged_df.get('Active_Legal_Proceedings'), errors='coerce')
merged_df['Single_Source_Dependency_Pct_num'] = pd.to_numeric(merged_df.get('Single_Source_Dependency_Pct'), errors='coerce')
merged_df['Avg_Days_Beyond_Terms_num'] = pd.to_numeric(merged_df.get('Avg_Days_Beyond_Terms'), errors='coerce')



bcs_inv_norm = 100 - normalize_0_100(merged_df['Business_Continuity_Score_num'])
legal_norm = normalize_0_100(merged_df['Active_Legal_Proceedings_num'])
dependency_norm = normalize_0_100(merged_df['Single_Source_Dependency_Pct_num'])

merged_df['Operational_Risk_Score'] = (
    bcs_inv_norm * 0.35 +
    legal_norm * 0.25 +
    dependency_norm * 0.25 
)

# --- Overall Supplier Risk Score (weighted average) ---
merged_df['Overall_Risk_Score'] = (
    merged_df['Financial_Risk_Score'] * 0.40 +
    merged_df['Sustainability_Risk_Score'] * 0.30 +
    merged_df['Operational_Risk_Score'] * 0.30
)

# --- Binary Target for ML ---
# Using multi-signal approach combining sustainability, payment behavior, and risk scores
# Percentile-based: top 25% of risk scores OR absolute thresholds
risk_75th = merged_df['Overall_Risk_Score'].quantile(0.75)
merged_df['Sanctions_Flag_str'] = merged_df.get('Sanctions_Flag', pd.Series('N', index=merged_df.index)).fillna('N')

merged_df['High_Risk'] = (
    (merged_df['Overall_Risk_Score'] > risk_75th) |
    (merged_df['Bankruptcy_Probability_Pct_num'] > 15) |
    (merged_df['Sanctions_Flag_str'].str.upper().isin(['YES', 'Y'])) |
    (merged_df['Carbon_Intensity'] > 2.5) # High carbon intensity
).astype(int)

print("Risk Score Summary:")
print(f"  Financial Risk - Mean: {merged_df['Financial_Risk_Score'].mean():.1f}, Std: {merged_df['Financial_Risk_Score'].std():.1f}")
print(f"  Sustainability Risk - Mean: {merged_df['Sustainability_Risk_Score'].mean():.1f}, Std: {merged_df['Sustainability_Risk_Score'].std():.1f}")
print(f"  Operational Risk - Mean: {merged_df['Operational_Risk_Score'].mean():.1f}, Std: {merged_df['Operational_Risk_Score'].std():.1f}")
print(f"  Overall Risk - Mean: {merged_df['Overall_Risk_Score'].mean():.1f}, Std: {merged_df['Overall_Risk_Score'].std():.1f}")
print(f"  75th percentile threshold: {risk_75th:.1f}")
print(f"\nHigh Risk Suppliers: {merged_df['High_Risk'].sum()} / {len(merged_df)} ({merged_df['High_Risk'].mean()*100:.1f}%)")

### ML Model: Supplier Disruption Risk Prediction

Using the simple scoring framework above, we will train a **Gradient Boosting Classifier** to predict supplier disruption risk using financial, sustainability, and operational features. This enables proactive identification of at-risk suppliers before disruptions occur.

> **Note:** We are not persisting the model at this stage, as it is currently being used only for calculation purposes. Persisting the model for reuse and future consumption can be considered as a potential enhancement on permanent systems. Eg. TDD

In [0]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer

# --- Define Features ---
numeric_features = [
    'Financial_Stress_Score_num', 'Bankruptcy_Probability_Pct_num',
    'Viability_Score_num', 'Paydex_Score_num', 'Avg_Days_Beyond_Terms_num',
    'Late_Payment_Pct_combined', 'Business_Continuity_Score_num',
    'Active_Legal_Proceedings_num', 'Single_Source_Dependency_Pct_num',
    'Total_Emissions', 'Carbon_Intensity',
    'Renewable Energy (%)', 'Annual Production',
    'Electricity (kWh)', 'Natural Gas (kWh)',
    'Landfill Waste (kg)', 'Recycled Waste (kg)',
    'total_spend', 'num_invoices', 'waste_ratio',
    'Days_Beyond_Combined'
]

categorical_features = ['Payment_Trend', 'Compliance_Risk_Level', 
                        'Operational_Risk_Level', 'Country_Risk_Tier', 'Carbon_Score']

# Filter to features that exist in the dataframe
numeric_features = [f for f in numeric_features if f in merged_df.columns]
categorical_features = [f for f in categorical_features if f in merged_df.columns]

# Remove numeric features that are entirely NaN (no useful signal)
X_numeric = merged_df[numeric_features].apply(pd.to_numeric, errors='coerce')
all_nan_cols = X_numeric.columns[X_numeric.isna().all()].tolist()
numeric_features = [f for f in numeric_features if f not in all_nan_cols]
X_numeric = X_numeric[numeric_features]

print(f"Numeric features ({len(numeric_features)}): {numeric_features}")
print(f"Categorical features ({len(categorical_features)}): {categorical_features}")
if all_nan_cols:
    print(f"Removed all-NaN features: {all_nan_cols}")

# --- Prepare Feature Matrix ---
# Numeric: impute with median (remaining NaN-heavy cols get 0)
num_imputer = SimpleImputer(strategy='median')
X_numeric_imputed = pd.DataFrame(
    num_imputer.fit_transform(X_numeric),
    columns=numeric_features,
    index=merged_df.index
)

# Categorical: impute with mode, then one-hot encode
X_categorical = merged_df[categorical_features].fillna('Unknown')
X_categorical_encoded = pd.get_dummies(X_categorical, columns=categorical_features, drop_first=False)

# Combine
X = pd.concat([X_numeric_imputed, X_categorical_encoded], axis=1)
y = merged_df['High_Risk'].values

print(f"\nFinal feature matrix shape: {X.shape}")
print(f"Target distribution: 0 (Low Risk)={sum(y==0)}, 1 (High Risk)={sum(y==1)}")

# --- Train/Test Split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# --- Scale Features ---
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test.index)

print(f"\nTraining set: {X_train_scaled.shape[0]} samples")
print(f"Test set: {X_test_scaled.shape[0]} samples")

In [0]:
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score

# --- Gradient Boosting Classifier ---
gbc_model = GradientBoostingClassifier(
    n_estimators=200, max_depth=5, random_state=42, learning_rate=0.1
)
gbc_model.fit(X_train_scaled, y_train)
y_pred_gbc = gbc_model.predict(X_test_scaled)
y_prob_gbc = gbc_model.predict_proba(X_test_scaled)[:, 1]

print("=" * 60)
print("GRADIENT BOOSTING CLASSIFIER")
print("=" * 60)
print(f"Accuracy: {accuracy_score(y_test, y_pred_gbc):.4f}")
print(f"ROC-AUC:  {roc_auc_score(y_test, y_prob_gbc):.4f}")
print(f"\nClassification Report:")
labels_present = sorted(set(y_test) | set(y_pred_gbc))
print(classification_report(y_test, y_pred_gbc, labels=labels_present, 
                            target_names=['Low Risk', 'High Risk'][:len(labels_present)]))

# --- Random Forest Classifier (comparison) ---
rf_model = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)
rf_model.fit(X_train_scaled, y_train)
y_pred_rf = rf_model.predict(X_test_scaled)
y_prob_rf = rf_model.predict_proba(X_test_scaled)[:, 1]

print("\n" + "=" * 60)
print("RANDOM FOREST CLASSIFIER (Comparison)")
print("=" * 60)
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"ROC-AUC:  {roc_auc_score(y_test, y_prob_rf):.4f}")
print(classification_report(y_test, y_pred_rf, labels=labels_present,
                            target_names=['Low Risk', 'High Risk'][:len(labels_present)]))

In [0]:
import matplotlib.pyplot as plt

# Get feature importances from Gradient Boosting model
feature_importance = pd.Series(gbc_model.feature_importances_, index=X.columns)
top_15 = feature_importance.nlargest(15).sort_values()

fig, ax = plt.subplots(figsize=(10, 8))
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(top_15)))
ax.barh(range(len(top_15)), top_15.values, color=colors, edgecolor='black', linewidth=0.5)
ax.set_yticks(range(len(top_15)))
ax.set_yticklabels(top_15.index, fontsize=10)
ax.set_xlabel('Feature Importance', fontsize=12)
ax.set_title('Top 15 Features - Gradient Boosting (Supplier Disruption Risk)', fontsize=13, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

### Actionable Supplier Recommendations

We will classify each supplier into one of four action categories based on combined risk, sustainability, and compliance signals:

| Category | Criteria | Action |
| --- | --- | --- |
| **Continue** (Green) | Low risk, good sustainability, compliant | Maintain relationship |
| **Monitor** (Yellow) | Moderate risk or medium carbon footprint | Increase oversight |
| **Improve** (Orange) | Elevated risk or poor sustainability | Engage for improvement |
| **Replace** (Red) | High risk, sanctions, or critical failures | Source alternatives |


Tip: The function below can be an ideal function to train a RAG Agent. 

In [0]:
def classify_supplier(row):
    """Classify supplier into Continue / Monitor / Improve / Replace."""
    # Replace (Red) - Critical issues
    if (row['Overall_Risk_Score'] > 70 or 
        row.get('Bankruptcy_Probability_Pct_num', 0) > 20 or
        str(row.get('Sanctions_Flag_str', 'N')).upper() in ['YES', 'Y']):
        return 'Replace'
    
    # Improve (Orange) - Needs attention
    if (row['Overall_Risk_Score'] > 50 or 
        row.get('Carbon_Score', 'B') == 'C' or
        row.get('Late_Payment_Pct_num', 0) > 30):
        return 'Improve'
    
    # Continue (Green) - Excellent performance
    if (row['Overall_Risk_Score'] < 30 and 
        row.get('Carbon_Score', 'B') == 'A' and
        str(row.get('Sanctions_Flag_str', 'N')).upper() not in ['YES', 'Y']):
        return 'Continue'
    
    # Monitor (Yellow) - Default for moderate cases
    return 'Monitor'

# Apply recommendation
merged_df['Recommendation'] = merged_df.apply(classify_supplier, axis=1)

# Display distribution
rec_dist = merged_df['Recommendation'].value_counts()
print("Supplier Recommendation Distribution:")
print("=" * 40)
for rec, count in rec_dist.items():
    pct = count / len(merged_df) * 100
    symbol = {'Continue': '\u2705', 'Monitor': '\u26a0\ufe0f', 'Improve': '\U0001f7e0', 'Replace': '\u274c'}[rec]
    print(f"  {symbol} {rec:10s}: {count:4d} suppliers ({pct:.1f}%)")

print(f"\nTotal suppliers analyzed: {len(merged_df)}")

Let us now look at some possible visualizations of the model outputs. Below charts are just suggestions. There are several insights that can be derived at this point. We will try to visualize insights in three different ways.

### 1. Supplier Intelligence Dashboard

In [0]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Supplier Risk & Sustainability Intelligence Dashboard', fontsize=14, fontweight='bold', y=0.98)

# Color mapping for recommendations
rec_colors = {'Continue': '#2ecc71', 'Monitor': '#f39c12', 'Improve': '#e67e22', 'Replace': '#e74c3c'}

# --- Plot 1: Financial Risk vs Sustainability Risk ---
ax1 = axes[0, 0]
for rec in ['Continue', 'Monitor', 'Improve', 'Replace']:
    mask = merged_df['Recommendation'] == rec
    if mask.any():
        ax1.scatter(
            merged_df.loc[mask, 'Financial_Risk_Score'],
            merged_df.loc[mask, 'Sustainability_Risk_Score'],
            c=rec_colors[rec], label=rec, alpha=0.6, s=30, edgecolors='black', linewidth=0.3
        )
ax1.set_xlabel('Financial Risk Score')
ax1.set_ylabel('Sustainability Risk Score')
ax1.set_title('Financial vs Sustainability Risk')
ax1.legend(loc='upper left', fontsize=8)
ax1.grid(alpha=0.3)

# --- Plot 2: Recommendation Distribution ---
ax2 = axes[0, 1]
rec_order = ['Continue', 'Monitor', 'Improve', 'Replace']
rec_counts = [merged_df['Recommendation'].value_counts().get(r, 0) for r in rec_order]
bar_colors = [rec_colors[r] for r in rec_order]
ax2.bar(rec_order, rec_counts, color=bar_colors, edgecolor='black', linewidth=0.5)
ax2.set_ylabel('Number of Suppliers')
ax2.set_title('Recommendation Distribution')
for i, v in enumerate(rec_counts):
    ax2.text(i, v + max(rec_counts)*0.02, str(v), ha='center', fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

# --- Plot 3: Carbon Score Distribution ---
ax3 = axes[1, 0]
carbon_counts = merged_df['Carbon_Score'].value_counts().sort_index()
carbon_colors = {'A': '#27ae60', 'B': '#f39c12', 'C': '#e74c3c'}
wedge_colors = [carbon_colors.get(s, '#95a5a6') for s in carbon_counts.index]
ax3.pie(carbon_counts.values, labels=[f"Score {s}\n({v} suppliers)" for s, v in carbon_counts.items()],
        colors=wedge_colors, autopct='%1.1f%%', startangle=90)
ax3.set_title('Carbon Score Distribution')

# --- Plot 4: Overall Risk Score Distribution ---
ax4 = axes[1, 1]
ax4.hist(merged_df['Overall_Risk_Score'].dropna(), bins=30, color='#3498db', 
         edgecolor='black', linewidth=0.5, alpha=0.7)
ax4.axvline(30, color='green', linestyle='--', linewidth=1.5, label='Continue threshold (30)')
ax4.axvline(50, color='orange', linestyle='--', linewidth=1.5, label='Improve threshold (50)')
ax4.axvline(70, color='red', linestyle='--', linewidth=1.5, label='Replace threshold (70)')
ax4.set_xlabel('Overall Risk Score')
ax4.set_ylabel('Number of Suppliers')
ax4.set_title('Overall Risk Score Distribution')
ax4.legend(fontsize=8)
ax4.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

### 2. ESG Reporting: Emissions Summary

Let us look at some more possible insights: Below are the aggregated emissions data supporting ESG disclosure requirements, identifying emission hotspots, and tracking progress toward sustainability targets.

In [0]:
print("=" * 60)
print("SCOPE 3 EMISSIONS SUMMARY")
print("=" * 60)

# Total emissions
total_scope3 = merged_df['Total_Emissions'].sum()
print(f"\nTotal Scope 3 Emissions: {total_scope3:,.0f} kg CO2 ({total_scope3/1000:,.0f} tonnes CO2)")
print(f"Average per supplier: {merged_df['Total_Emissions'].mean():,.0f} kg CO2")
print(f"Median per supplier: {merged_df['Total_Emissions'].median():,.0f} kg CO2")

# Emissions breakdown by category
print(f"\nEmissions Breakdown by Category:")
print("-" * 40)
categories = {
    'Electricity (adj.)': merged_df['Electricity_Emissions'].sum(),
    'Natural Gas': merged_df['Gas_Emissions'].sum(),
    'Truck Transport': merged_df['Truck_Emissions'].sum(),
    'Ship Transport': merged_df['Ship_Emissions'].sum(),
    'Landfill Waste': merged_df['Landfill_Emissions'].sum(),
    'Recycling': merged_df['Recycling_Emissions'].sum()
}
for cat, val in sorted(categories.items(), key=lambda x: -x[1]):
    pct = val / total_scope3 * 100
    print(f"  {cat:20s}: {val:>12,.0f} kg CO2 ({pct:.1f}%)")

# Carbon intensity stats
print(f"\nCarbon Intensity Statistics:")
print(f"  Mean:   {merged_df['Carbon_Intensity'].mean():.2f} kg CO2/unit")
print(f"  Median: {merged_df['Carbon_Intensity'].median():.2f} kg CO2/unit")
print(f"  Std:    {merged_df['Carbon_Intensity'].std():.2f} kg CO2/unit")
print(f"  Min:    {merged_df['Carbon_Intensity'].min():.2f} kg CO2/unit")
print(f"  Max:    {merged_df['Carbon_Intensity'].max():.2f} kg CO2/unit")

# ESG threshold compliance
score_a_pct = (merged_df['Carbon_Score'] == 'A').sum() / len(merged_df) * 100
print(f"\nESG Compliance:")
print(f"  Suppliers meeting ESG threshold (Score A): {(merged_df['Carbon_Score'] == 'A').sum()} ({score_a_pct:.1f}%)")
print(f"  Suppliers at medium risk (Score B): {(merged_df['Carbon_Score'] == 'B').sum()}")
print(f"  Suppliers at high risk (Score C): {(merged_df['Carbon_Score'] == 'C').sum()}")

# Top 10 highest emitting suppliers
print(f"\nTop 10 Highest-Emitting Suppliers:")
top_emitters = merged_df.nlargest(10, 'Total_Emissions')[[
    'Supplier Name', 'Total_Emissions', 'Carbon_Intensity', 'Carbon_Score', 'Recommendation'
]].copy()
top_emitters['Total_Emissions'] = top_emitters['Total_Emissions'].round(0)
top_emitters['Carbon_Intensity'] = top_emitters['Carbon_Intensity'].round(2)
display(top_emitters)

### Top Priority Actions

Final actionable intelligence identifying suppliers requiring immediate attention, quantifying financial exposure at risk, and estimating potential CO2 reduction from recommended improvements.

In [0]:
# --- Top 10 Suppliers to REPLACE ---
replace_suppliers = merged_df[merged_df['Recommendation'] == 'Replace'].nlargest(
    10, 'Overall_Risk_Score'
)[['Supplier Name', 'Overall_Risk_Score', 'Financial_Risk_Score', 
   'Sustainability_Risk_Score', 'Carbon_Score', 'Total_Emissions', 'total_spend']].copy()
replace_suppliers.columns = ['Supplier', 'Overall Risk', 'Financial Risk', 
                              'Sustainability Risk', 'Carbon Score', 'Emissions (kg CO2)', 'Total Spend']
replace_suppliers['Overall Risk'] = replace_suppliers['Overall Risk'].round(1)
replace_suppliers['Financial Risk'] = replace_suppliers['Financial Risk'].round(1)
replace_suppliers['Sustainability Risk'] = replace_suppliers['Sustainability Risk'].round(1)
replace_suppliers['Emissions (kg CO2)'] = replace_suppliers['Emissions (kg CO2)'].round(0)
replace_suppliers['Total Spend'] = replace_suppliers['Total Spend'].round(2)

print("\u274c TOP 10 SUPPLIERS TO REPLACE (Highest Combined Risk)")
print("=" * 70)
display(replace_suppliers)

# --- Top 10 Suppliers to IMPROVE ---
improve_suppliers = merged_df[merged_df['Recommendation'] == 'Improve'].nlargest(
    10, 'Total_Emissions'
)[['Supplier Name', 'Overall_Risk_Score', 'Carbon_Intensity', 
   'Carbon_Score', 'Total_Emissions', 'Renewable Energy (%)', 'total_spend']].copy()
improve_suppliers.columns = ['Supplier', 'Overall Risk', 'Carbon Intensity', 
                              'Carbon Score', 'Emissions (kg CO2)', 'Renewable %', 'Total Spend']
improve_suppliers['Overall Risk'] = improve_suppliers['Overall Risk'].round(1)
improve_suppliers['Carbon Intensity'] = improve_suppliers['Carbon Intensity'].round(2)
improve_suppliers['Emissions (kg CO2)'] = improve_suppliers['Emissions (kg CO2)'].round(0)
improve_suppliers['Total Spend'] = improve_suppliers['Total Spend'].round(2)

print("\n\n\U0001f7e0 TOP 10 SUPPLIERS TO IMPROVE (Clear Improvement Paths)")
print("=" * 70)
display(improve_suppliers)

# --- Summary Statistics ---
print("\n\n" + "=" * 70)
print("EXECUTIVE SUMMARY")
print("=" * 70)

replace_mask = merged_df['Recommendation'] == 'Replace'
improve_mask = merged_df['Recommendation'] == 'Improve'

total_exposure_replace = merged_df.loc[replace_mask, 'total_spend'].sum()
total_emissions_replace = merged_df.loc[replace_mask, 'Total_Emissions'].sum()
total_emissions_improve = merged_df.loc[improve_mask, 'Total_Emissions'].sum()

# Potential CO2 reduction: if "Improve" suppliers increase renewable to 50%
current_renewable_avg = merged_df.loc[improve_mask, 'Renewable Energy (%)'].mean()
potential_reduction = total_emissions_improve * 0.20  # Conservative 20% reduction estimate

print(f"\n  Financial Exposure at Risk (Replace suppliers):")
print(f"    Total spend with Replace suppliers: ${total_exposure_replace:,.2f}")
print(f"    Number of suppliers to replace: {replace_mask.sum()}")

print(f"\n  Emissions Impact:")
print(f"    CO2 from Replace suppliers: {total_emissions_replace:,.0f} kg ({total_emissions_replace/1000:,.0f} tonnes)")
print(f"    CO2 from Improve suppliers: {total_emissions_improve:,.0f} kg ({total_emissions_improve/1000:,.0f} tonnes)")
print(f"    Potential CO2 reduction (20% improvement): {potential_reduction:,.0f} kg ({potential_reduction/1000:,.0f} tonnes)")

print(f"\n  Recommendation Breakdown:")
for rec in ['Continue', 'Monitor', 'Improve', 'Replace']:
    count = (merged_df['Recommendation'] == rec).sum()
    pct = count / len(merged_df) * 100
    print(f"    {rec:10s}: {count:4d} ({pct:5.1f}%)")

### Data Model for SAP Datasphere & SAC

We will now create a star schema data model with dimension and fact tables to enable:
- Data modeling in **SAP Datasphere**
- Story building in **SAP Analytics Cloud (SAC)**

### Schema: `cnr_catalog.<schema_name>`

| Table | Type | Description |
| --- | --- | --- |
| `dp_dim_supplier` | Dimension | Supplier master data (ID, name, location, size) |
| `dp_dim_recommendation` | Dimension | Action categories (Continue/Monitor/Improve/Replace) |
| `dp_dim_carbon_score` | Dimension | Carbon score classifications (A/B/C) |
| `dp_fact_risk_assessment` | Fact | Risk scores and classifications per supplier |
| `dp_fact_carbon_emissions` | Fact | Scope 3 emissions breakdown per supplier |
| `dp_fact_supplier_spend` | Fact | Aggregated procurement spend per supplier |

Let us create a schema within the cnr_catalog with your own ID. Please make sure to use your own schema to persist your data tables.

# &#x270D;
# CREATE YOUR OWN SCHEMA
In order to isolate the created data assets, we create a catalog within Databricks and a respective schema within the catalog. Please replace the values `<CATALOG_NAME>` and `<SCHEMA_NAME>` with the specific values that match our use case and group. You can find the correct names by checking the **Unity Catalog** and look for the specific catalog and schema names: here replace _<USERID> with your appropriate user id given to you. i.e. Your schema name will be cnr_AC2XXXXXXX01, cnr_AC2XXXXXXX02 and so on. 

In [0]:
%sql
SET CATALOG cnr_catalog;
CREATE SCHEMA IF NOT EXISTS cnr_<USERID>;
USE SCHEMA cnr_<USERID>;

In [0]:
# =============================================================================
# DIMENSION TABLES
# =============================================================================

# --- 1. dp_dim_supplier (Supplier Master Data) ---
dim_supplier = merged_df[['Supplier Name', 'DunsNumber', 'MappedSupplierName']].copy()
dim_supplier = dim_supplier.rename(columns={
    'Supplier Name': 'Supplier',
    'DunsNumber': 'duns_number',
    'MappedSupplierName': 'supplier_name'
})

# Add location and business attributes from risk data
for col, new_col in [('CityName', 'city'), ('Region', 'region'), ('Country', 'country'),
                      ('Years_In_Business', 'years_in_business'), ('Employee_Count', 'employee_count')]:
    if col in merged_df.columns:
        dim_supplier[new_col] = merged_df[col].values
    else:
        dim_supplier[new_col] = None

dim_supplier = dim_supplier.drop_duplicates(subset=['Supplier']).reset_index(drop=True)

# Write to Unity Catalog
spark_dim_supplier = spark.createDataFrame(dim_supplier)
spark_dim_supplier.write.mode('overwrite').saveAsTable('dp_dim_supplier')

# Set NOT NULL on PK column and add Primary Key constraint
spark.sql("ALTER TABLE dp_dim_supplier ALTER COLUMN Supplier SET NOT NULL")
spark.sql("ALTER TABLE dp_dim_supplier DROP CONSTRAINT IF EXISTS pk_supplier")
spark.sql("ALTER TABLE dp_dim_supplier ADD CONSTRAINT pk_supplier PRIMARY KEY (Supplier)")




print(f"\u2705 dp_dim_supplier: {dim_supplier.shape[0]} rows")
print(f"   PK: supplier_id")
print(f"   Columns: {list(dim_supplier.columns)}")

# --- 2. dp_dim_recommendation (Action Categories) ---
dim_recommendation = pd.DataFrame({
    'recommendation_id': [1, 2, 3, 4],
    'recommendation_name': ['Continue', 'Monitor', 'Improve', 'Replace'],
    'description': [
        'Low risk, good sustainability - maintain relationship',
        'Moderate risk or medium carbon footprint - increase oversight',
        'Elevated risk or poor sustainability - engage for improvement',
        'High risk, sanctions, or critical failures - source alternatives'
    ],
    'action_priority': [4, 3, 2, 1],
    'color_code': ['#2ecc71', '#f39c12', '#e67e22', '#e74c3c']
})

spark_dim_rec = spark.createDataFrame(dim_recommendation)
spark_dim_rec.write.mode('overwrite').saveAsTable('dp_dim_recommendation')

spark.sql("ALTER TABLE dp_dim_recommendation ALTER COLUMN recommendation_id SET NOT NULL")
spark.sql("ALTER TABLE dp_dim_recommendation DROP CONSTRAINT IF EXISTS dim_recommendation")
spark.sql("ALTER TABLE dp_dim_recommendation ADD CONSTRAINT dim_recommendation PRIMARY KEY (recommendation_id)")

print(f"\n\u2705 dp_dim_recommendation: {dim_recommendation.shape[0]} rows")
print(f"   PK: recommendation_id")

# --- 3. dp_dim_carbon_score (Carbon Score Classifications) ---
dim_carbon_score = pd.DataFrame({
    'carbon_score_id': ['A', 'B', 'C'],
    'score_label': ['Low Carbon', 'Medium Carbon', 'High Carbon'],
    'description': [
        'Carbon Intensity < 1.5 kg CO2/unit - meets ESG target',
        'Carbon Intensity 1.5-3.0 kg CO2/unit - moderate emissions',
        'Carbon Intensity > 3.0 kg CO2/unit - exceeds ESG threshold'
    ],
    'intensity_threshold_min': [0.0, 1.5, 3.0],
    'intensity_threshold_max': [1.5, 3.0, 999.0]
})

spark_dim_cs = spark.createDataFrame(dim_carbon_score)
spark_dim_cs.write.mode('overwrite').saveAsTable('dp_dim_carbon_score')

spark.sql("ALTER TABLE dp_dim_carbon_score ALTER COLUMN carbon_score_id SET NOT NULL")
spark.sql("ALTER TABLE dp_dim_carbon_score DROP CONSTRAINT IF EXISTS pk_carbon_score")

spark.sql("ALTER TABLE dp_dim_carbon_score ADD CONSTRAINT pk_carbon_score PRIMARY KEY (carbon_score_id)")

print(f"\n\u2705 dp_dim_carbon_score: {dim_carbon_score.shape[0]} rows")
print(f"   PK: carbon_score_id")

In [0]:
# =============================================================================
# FACT TABLES
# =============================================================================

# --- Map recommendation names to IDs ---
rec_map = {'Continue': 1, 'Monitor': 2, 'Improve': 3, 'Replace': 4}

# --- 4. dp_fact_risk_assessment ---
fact_risk = merged_df[['Supplier Name']].copy()
fact_risk = fact_risk.rename(columns={'Supplier Name': 'supplier_id'})
fact_risk['financial_risk_score'] = merged_df['Financial_Risk_Score'].round(2)
fact_risk['sustainability_risk_score'] = merged_df['Sustainability_Risk_Score'].round(2)
fact_risk['operational_risk_score'] = merged_df['Operational_Risk_Score'].round(2)
fact_risk['overall_risk_score'] = merged_df['Overall_Risk_Score'].round(2)
fact_risk['high_risk_flag'] = merged_df['High_Risk'].astype(int)
fact_risk['recommendation_id'] = merged_df['Recommendation'].map(rec_map)
fact_risk['carbon_score_id'] = merged_df['Carbon_Score']
fact_risk['bankruptcy_probability_pct'] = pd.to_numeric(merged_df['Bankruptcy_Probability_Pct'], errors='coerce').round(2)
fact_risk['viability_score'] = pd.to_numeric(merged_df['Viability_Score'], errors='coerce')
fact_risk['paydex_score'] = pd.to_numeric(merged_df['Paydex_Score'], errors='coerce')
fact_risk['financial_stress_score'] = pd.to_numeric(merged_df['Financial_Stress_Score'], errors='coerce').round(2)
fact_risk['business_continuity_score'] = pd.to_numeric(merged_df['Business_Continuity_Score'], errors='coerce').round(2)

fact_risk = fact_risk.drop_duplicates(subset=['supplier_id']).reset_index(drop=True)

spark_fact_risk = spark.createDataFrame(fact_risk)
spark_fact_risk.write.mode('overwrite').saveAsTable('dp_fact_risk_assessment')

# Set NOT NULL on PK, then add PK and FK constraints
spark.sql("ALTER TABLE dp_fact_risk_assessment ALTER COLUMN supplier_id SET NOT NULL")
spark.sql("ALTER TABLE dp_fact_risk_assessment ADD CONSTRAINT pk_fact_risk PRIMARY KEY (supplier_id)")
spark.sql("""
    ALTER TABLE dp_fact_risk_assessment
    ADD CONSTRAINT fk_risk_supplier FOREIGN KEY (supplier_id)
    REFERENCES dp_dim_supplier(Supplier)
""")
spark.sql("""
    ALTER TABLE dp_fact_risk_assessment
    ADD CONSTRAINT fk_risk_recommendation FOREIGN KEY (recommendation_id)
    REFERENCES dp_dim_recommendation(recommendation_id)
""")
spark.sql("""
    ALTER TABLE dp_fact_risk_assessment
    ADD CONSTRAINT fk_risk_carbon_score FOREIGN KEY (carbon_score_id)
    REFERENCES dp_dim_carbon_score(carbon_score_id)
""")

print(f"\u2705 dp_fact_risk_assessment: {fact_risk.shape[0]} rows")
print(f"   PK: supplier_id")
print(f"   FK: supplier_id \u2192 dp_dim_supplier")
print(f"   FK: recommendation_id \u2192 dp_dim_recommendation")
print(f"   FK: carbon_score_id \u2192 dp_dim_carbon_score")

# --- 5. dp_fact_carbon_emissions ---
fact_carbon = merged_df[['Supplier Name']].copy()
fact_carbon = fact_carbon.rename(columns={'Supplier Name': 'supplier_id'})
fact_carbon['total_emissions_kg'] = merged_df['Total_Emissions'].round(2)
fact_carbon['carbon_intensity'] = merged_df['Carbon_Intensity'].round(4)
fact_carbon['carbon_score_id'] = merged_df['Carbon_Score']
fact_carbon['electricity_emissions_kg'] = merged_df['Electricity_Emissions'].round(2)
fact_carbon['gas_emissions_kg'] = merged_df['Gas_Emissions'].round(2)
fact_carbon['truck_emissions_kg'] = merged_df['Truck_Emissions'].round(2)
fact_carbon['ship_emissions_kg'] = merged_df['Ship_Emissions'].round(2)
fact_carbon['landfill_emissions_kg'] = merged_df['Landfill_Emissions'].round(2)
fact_carbon['recycling_emissions_kg'] = merged_df['Recycling_Emissions'].round(2)
fact_carbon['renewable_energy_pct'] = merged_df['Renewable Energy (%)']
fact_carbon['annual_production_units'] = merged_df['Annual Production']
fact_carbon['electricity_kwh'] = merged_df['Electricity (kWh)']
fact_carbon['natural_gas_kwh'] = merged_df['Natural Gas (kWh)']
fact_carbon['truck_distance_km'] = merged_df['Truck Distance (km)']
fact_carbon['ship_distance_km'] = merged_df['Ship Distance (km)']
fact_carbon['landfill_waste_kg'] = merged_df['Landfill Waste (kg)']
fact_carbon['recycled_waste_kg'] = merged_df['Recycled Waste (kg)']

fact_carbon = fact_carbon.drop_duplicates(subset=['supplier_id']).reset_index(drop=True)

spark_fact_carbon = spark.createDataFrame(fact_carbon)
spark_fact_carbon.write.mode('overwrite').saveAsTable('dp_fact_carbon_emissions')

spark.sql("ALTER TABLE dp_fact_carbon_emissions ALTER COLUMN supplier_id SET NOT NULL")
spark.sql("ALTER TABLE dp_fact_carbon_emissions ADD CONSTRAINT pk_fact_carbon PRIMARY KEY (supplier_id)")
spark.sql("""
    ALTER TABLE dp_fact_carbon_emissions
    ADD CONSTRAINT fk_carbon_supplier FOREIGN KEY (supplier_id)
    REFERENCES dp_dim_supplier(Supplier)
""")
spark.sql("""
    ALTER TABLE dp_fact_carbon_emissions
    ADD CONSTRAINT fk_carbon_score FOREIGN KEY (carbon_score_id)
    REFERENCES dp_dim_carbon_score(carbon_score_id)
""")

print(f"\n\u2705 dp_fact_carbon_emissions: {fact_carbon.shape[0]} rows")
print(f"   PK: supplier_id")
print(f"   FK: supplier_id \u2192 dp_dim_supplier")
print(f"   FK: carbon_score_id \u2192 dp_dim_carbon_score")

# --- 6. dp_fact_supplier_spend ---
fact_spend = merged_df[['Supplier Name', 'total_spend', 'num_invoices',
                         'avg_order_value', 'num_unique_items']].copy()
fact_spend = fact_spend.rename(columns={
    'Supplier Name': 'supplier_id',
    'total_spend': 'total_spend_amount',
    'num_invoices': 'invoice_count',
    'avg_order_value': 'avg_order_value',
    'num_unique_items': 'unique_items_count'
})
fact_spend['invoice_count'] = fact_spend['invoice_count'].fillna(0).astype(int)
fact_spend['unique_items_count'] = fact_spend['unique_items_count'].fillna(0).astype(int)
fact_spend['total_spend_amount'] = fact_spend['total_spend_amount'].round(2)
fact_spend['avg_order_value'] = fact_spend['avg_order_value'].round(2)

fact_spend = fact_spend.drop_duplicates(subset=['supplier_id']).reset_index(drop=True)

spark_fact_spend = spark.createDataFrame(fact_spend)
spark_fact_spend.write.mode('overwrite').saveAsTable('dp_fact_supplier_spend')

spark.sql("ALTER TABLE dp_fact_supplier_spend ALTER COLUMN supplier_id SET NOT NULL")
spark.sql("ALTER TABLE dp_fact_supplier_spend ADD CONSTRAINT pk_fact_spend PRIMARY KEY (supplier_id)")
spark.sql("""
    ALTER TABLE dp_fact_supplier_spend
    ADD CONSTRAINT fk_spend_supplier FOREIGN KEY (supplier_id)
    REFERENCES dp_dim_supplier(Supplier)
""")

print(f"\n\u2705 dp_fact_supplier_spend: {fact_spend.shape[0]} rows")
print(f"   PK: supplier_id")
print(f"   FK: supplier_id \u2192 dp_dim_supplier")

In [0]:
# =============================================================================
# VERIFY DATA MODEL
# =============================================================================

print("=" * 70)
print("DATA MODEL SUMMARY - SAP Datasphere / SAC")
print("=" * 70)

tables = [
    'dp_dim_supplier',
    'dp_dim_recommendation',
    'dp_dim_carbon_score',
    'dp_fact_risk_assessment',
    'dp_fact_carbon_emissions',
    'dp_fact_supplier_spend'
]

print("\n\u2500" * 70)
print(f"{'Table':<50} {'Rows':>8} {'Cols':>6}")
print("\u2500" * 70)
for t in tables:
    df_check = spark.read.table(t)
    row_count = df_check.count()
    col_count = len(df_check.columns)
    short_name = t.split('.')[-1]
    table_type = 'DIM' if 'dim' in short_name else 'FACT'
    print(f"  [{table_type}] {short_name:<44} {row_count:>8,} {col_count:>6}")
print("\u2500" * 70)

# Print relationships
print("\n\U0001f517 RELATIONSHIPS (Foreign Keys):")
print("-" * 70)
relationships = [
    ('dp_fact_risk_assessment.supplier_id', 'dp_dim_supplier.supplier_id'),
    ('dp_fact_risk_assessment.recommendation_id', 'dp_dim_recommendation.recommendation_id'),
    ('dp_fact_risk_assessment.carbon_score_id', 'dp_dim_carbon_score.carbon_score_id'),
    ('dp_fact_carbon_emissions.supplier_id', 'p_dim_supplier.supplier_id'),
    ('dp_fact_carbon_emissions.carbon_score_id', 'dp_dim_carbon_score.carbon_score_id'),
    ('dp_fact_supplier_spend.supplier_id', 'dp_dim_supplier.supplier_id'),
]
for fk_col, pk_col in relationships:
    print(f"  {fk_col:<50} \u2192 {pk_col}")

print("\n\U0001f4ca SAP Datasphere Modeling Notes:")
print("  - Import tables as Remote Tables via Databricks connection")
print("  - Map DIM tables as 'Dimension' semantic type")
print("  - Map FACT tables as 'Analytical Dataset' semantic type")
print("  - Create associations using the FK relationships above")
print("  - In SAC, build stories using the fact tables as data sources")

In [0]:
# =============================================================================
# TIME DIMENSION TABLE - For Trend Analysis in SAP Datasphere / SAC
# =============================================================================

from datetime import date, timedelta

# Generate a date range covering the invoice data period (2025-01 to 2027-12)
start_date = date(2025, 1, 1)
end_date = date(2027, 12, 31)
date_range = pd.date_range(start=start_date, end=end_date, freq='D')

# Build the time dimension
dim_time = pd.DataFrame({
    'date_id': date_range.strftime('%Y%m%d').astype(int),
    'full_date': date_range.date,
    'year': date_range.year,
    'quarter': date_range.quarter,
    'quarter_name': ['Q' + str(q) for q in date_range.quarter],
    'month': date_range.month,
    'month_name': date_range.strftime('%B'),
    'month_short': date_range.strftime('%b'),
    'week_of_year': date_range.isocalendar().week.astype(int),
    'day_of_month': date_range.day,
    'day_of_week': date_range.dayofweek + 1,  # 1=Monday, 7=Sunday
    'day_name': date_range.strftime('%A'),
    'is_weekend': (date_range.dayofweek >= 5).astype(int),
    'year_month': date_range.strftime('%Y-%m'),
    'year_quarter': date_range.strftime('%Y') + '-Q' + date_range.quarter.astype(str),
    'fiscal_year': np.where(date_range.month >= 10, date_range.year + 1, date_range.year)
})

print(f"Time dimension: {dim_time.shape[0]} rows ({start_date} to {end_date})")
print(f"Columns: {list(dim_time.columns)}")

# Write to Unity Catalog (drop constraints first for idempotency)
try:
    spark.sql("ALTER TABLE dp_dim_time DROP CONSTRAINT IF EXISTS pk_dim_time")
except:
    pass

spark_dim_time = spark.createDataFrame(dim_time)
spark_dim_time.write.mode('overwrite').saveAsTable('dp_dim_time')

spark.sql("ALTER TABLE dp_dim_time ALTER COLUMN date_id SET NOT NULL")
spark.sql("ALTER TABLE dp_dim_time ADD CONSTRAINT pk_dim_time PRIMARY KEY (date_id)")

print(f"\n\u2705 dp_dim_time: {dim_time.shape[0]} rows")
print(f"   PK: date_id")
print(f"   Range: {start_date} to {end_date}")
print(f"   Supports: yearly, quarterly, monthly, weekly trend analysis")
print(f"   Fiscal year: Oct-Sep cycle")

# --- Update fact_supplier_spend with date keys for time-based joins ---
invoice_dates = invoices_df.groupby('Supplier').agg(
    first_order_date=('PurchaseOrderDate', 'min'),
    last_order_date=('PurchaseOrderDate', 'max'),
    first_invoice_date=('PostingDate', 'min'),
    last_invoice_date=('PostingDate', 'max')
).reset_index()
invoice_dates['supplier_key'] = invoice_dates['Supplier'].astype(str).str.strip()

# Convert dates to date_id format (YYYYMMDD integer)
for col in ['first_order_date', 'last_order_date', 'first_invoice_date', 'last_invoice_date']:
    invoice_dates[col + '_id'] = pd.to_datetime(invoice_dates[col]).dt.strftime('%Y%m%d').astype(float)

# Merge date keys into fact_spend
fact_spend_with_dates = fact_spend.merge(
    invoice_dates[['supplier_key', 'first_order_date_id', 'last_order_date_id',
                    'first_invoice_date_id', 'last_invoice_date_id']].rename(
        columns={'supplier_key': 'supplier_id'}
    ),
    on='supplier_id', how='left'
)

# Drop existing constraints before overwriting
try:
    spark.sql("ALTER TABLE dp_fact_supplier_spend DROP CONSTRAINT IF EXISTS pk_fact_spend")
    spark.sql("ALTER TABLE dp_fact_supplier_spend DROP CONSTRAINT IF EXISTS fk_spend_supplier")
except:
    pass

# Overwrite fact table with new schema (includes date keys)
spark_fact_spend_dates = spark.createDataFrame(fact_spend_with_dates)
spark_fact_spend_dates.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable('dp_fact_supplier_spend')

spark.sql("ALTER TABLE dp_fact_supplier_spend ALTER COLUMN supplier_id SET NOT NULL")
spark.sql("ALTER TABLE dp_fact_supplier_spend ADD CONSTRAINT pk_fact_spend PRIMARY KEY (supplier_id)")
spark.sql("""
    ALTER TABLE dp_fact_supplier_spend
    ADD CONSTRAINT fk_spend_supplier FOREIGN KEY (supplier_id)
    REFERENCES dp_dim_supplier(Supplier)
""")

print(f"\n\u2705 dp_fact_supplier_spend updated with date keys")
print(f"   New columns: first_order_date_id, last_order_date_id, first_invoice_date_id, last_invoice_date_id")
print(f"   These join to dp_dim_time.date_id for trend analysis")

In [0]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(1, 1, figsize=(16, 11))
ax.set_xlim(0, 16)
ax.set_ylim(0, 11)
ax.axis('off')
ax.set_title('Supplier Risk & Sustainability Data Model\n(Star Schema for SAP Datasphere / SAC)', 
             fontsize=14, fontweight='bold', pad=20)

# --- Define table positions and content ---
tables = {
    'dp_dim_supplier': {
        'pos': (1.5, 5.5), 'w': 3.5, 'h': 2.5, 'type': 'DIM',
        'cols': ['\u2b50 supplier_id (PK)', 'duns_number', 'supplier_name',
                 'city, region, country', 'years_in_business', 'employee_count']
    },
    'dp_dim_time': {
        'pos': (6.5, 9.0), 'w': 3.2, 'h': 2.0, 'type': 'DIM',
        'cols': ['\u2b50 date_id (PK)', 'full_date, year, quarter',
                 'month, week_of_year', 'fiscal_year, is_weekend']
    },
    'dp_dim_recommendation': {
        'pos': (12.0, 8.5), 'w': 3.2, 'h': 1.8, 'type': 'DIM',
        'cols': ['\u2b50 recommendation_id (PK)', 'recommendation_name',
                 'description, action_priority']
    },
    'dp_dim_carbon_score': {
        'pos': (12.0, 5.5), 'w': 3.2, 'h': 1.8, 'type': 'DIM',
        'cols': ['\u2b50 carbon_score_id (PK)', 'score_label, description',
                 'intensity_threshold_min/max']
    },
    'dp_fact_risk_assessment': {
        'pos': (6.5, 5.5), 'w': 4.0, 'h': 2.8, 'type': 'FACT',
        'cols': ['\u2b50 supplier_id (PK/FK)', '\u2192 recommendation_id (FK)',
                 '\u2192 carbon_score_id (FK)', 'financial_risk_score',
                 'sustainability_risk_score', 'operational_risk_score',
                 'overall_risk_score, high_risk_flag']
    },
    'dp_fact_carbon_emissions': {
        'pos': (6.5, 1.5), 'w': 4.0, 'h': 2.5, 'type': 'FACT',
        'cols': ['\u2b50 supplier_id (PK/FK)', '\u2192 carbon_score_id (FK)',
                 'total_emissions_kg, carbon_intensity',
                 'electricity/gas/truck/ship_emissions',
                 'renewable_energy_pct, annual_production']
    },
    'dp_fact_supplier_spend': {
        'pos': (1.5, 1.5), 'w': 3.5, 'h': 2.2, 'type': 'FACT',
        'cols': ['\u2b50 supplier_id (PK/FK)', 'total_spend_amount',
                 'invoice_count, avg_order_value',
                 'first/last_order_date_id (FK\u2192time)']
}
}
# --- Draw tables ---
for name, info in tables.items():
    x, y = info['pos']
    w, h = info['w'], info['h']
    
    # Color based on type
    if info['type'] == 'DIM':
        header_color = '#3498db'
        body_color = '#ebf5fb'
    else:
        header_color = '#e74c3c'
        body_color = '#fdedec'
    
    # Draw box
    rect = FancyBboxPatch((x, y - h), w, h, boxstyle='round,pad=0.05',
                          facecolor=body_color, edgecolor='#2c3e50', linewidth=1.5)
    ax.add_patch(rect)
    
    # Header bar
    header_rect = FancyBboxPatch((x, y - 0.45), w, 0.45, boxstyle='round,pad=0.02',
                                  facecolor=header_color, edgecolor='#2c3e50', linewidth=1.5)
    ax.add_patch(header_rect)
    
    # Table name
    short_name = name.replace('dp_', '')
    ax.text(x + w/2, y - 0.22, short_name, ha='center', va='center',
            fontsize=8.5, fontweight='bold', color='white')
    
    # Column list
    for i, col in enumerate(info['cols']):
        ax.text(x + 0.15, y - 0.65 - i * 0.28, col, ha='left', va='center',
                fontsize=6.5, color='#2c3e50')

# --- Draw relationships (arrows) ---
arrow_style = dict(arrowstyle='->', color='#7f8c8d', linewidth=1.5,
                   connectionstyle='arc3,rad=0.1')

# fact_risk_assessment → dim_supplier
ax.annotate('', xy=(5.0, 6.2), xytext=(6.5, 6.5), arrowprops=arrow_style)
# fact_risk_assessment → dim_recommendation
ax.annotate('', xy=(12.0, 9.0), xytext=(10.5, 7.5),
            arrowprops=dict(arrowstyle='->', color='#7f8c8d', linewidth=1.5, connectionstyle='arc3,rad=-0.1'))
# fact_risk_assessment → dim_carbon_score
ax.annotate('', xy=(12.0, 6.5), xytext=(10.5, 6.2), arrowprops=arrow_style)
# fact_carbon_emissions → dim_supplier
ax.annotate('', xy=(3.5, 3.5), xytext=(6.5, 2.5),
            arrowprops=dict(arrowstyle='->', color='#7f8c8d', linewidth=1.5, connectionstyle='arc3,rad=0.2'))
# fact_carbon_emissions → dim_carbon_score
ax.annotate('', xy=(12.0, 5.5), xytext=(10.5, 3.5),
            arrowprops=dict(arrowstyle='->', color='#7f8c8d', linewidth=1.5, connectionstyle='arc3,rad=-0.2'))
# fact_supplier_spend → dim_supplier
ax.annotate('', xy=(2.5, 3.7), xytext=(2.5, 5.5),
            arrowprops=dict(arrowstyle='->', color='#7f8c8d', linewidth=1.5, connectionstyle='arc3,rad=0'))
# fact_supplier_spend → dim_time
ax.annotate('', xy=(7.5, 9.0), xytext=(4.0, 3.7),
            arrowprops=dict(arrowstyle='->', color='#27ae60', linewidth=1.5, linestyle='dashed', connectionstyle='arc3,rad=-0.3'))

# --- Legend ---
legend_elements = [
    mpatches.Patch(facecolor='#ebf5fb', edgecolor='#3498db', linewidth=2, label='Dimension Table'),
    mpatches.Patch(facecolor='#fdedec', edgecolor='#e74c3c', linewidth=2, label='Fact Table'),
    plt.Line2D([0], [0], color='#7f8c8d', linewidth=1.5, label='Foreign Key (solid)'),
    plt.Line2D([0], [0], color='#27ae60', linewidth=1.5, linestyle='dashed', label='Date FK (dashed)'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9, framealpha=0.9)

# Schema annotation
ax.text(0.2, 10.7, 'Schema: np   |    Prefix: dp_*    |    7 Tables (4 DIM + 3 FACT)',
        fontsize=9, style='italic', color='#555555')

plt.tight_layout()
plt.show()

## Data Model Documentation

**Schema:** `cnr_catalog.default` | **Prefix:** `dp_*` | **Architecture:** Star Schema

---

### Dimension Tables

#### `dp_dim_supplier` — Supplier Master Data

| Column | Type | Constraint | Description |
| --- | --- | --- | --- |
| supplier_id | STRING | **PK, NOT NULL** | Unique supplier identifier (10-digit zero-padded) |
| duns_number | STRING | | D&B DUNS number for external risk data linkage |
| supplier_name | STRING | | Human-readable supplier name |
| city | STRING | | Supplier city |
| region | STRING | | Supplier region/state |
| country | STRING | | Supplier country code |
| years_in_business | BIGINT | | Years of operation |
| employee_count | BIGINT | | Number of employees |

---

#### `dp_dim_time` — Time/Calendar Dimension

| Column | Type | Constraint | Description |
| --- | --- | --- | --- |
| date_id | BIGINT | **PK, NOT NULL** | Date key in YYYYMMDD format |
| full_date | DATE | | Calendar date |
| year | BIGINT | | Calendar year (2025–2027) |
| quarter | BIGINT | | Quarter number (1–4) |
| quarter_name | STRING | | Quarter label (Q1–Q4) |
| month | BIGINT | | Month number (1–12) |
| month_name | STRING | | Full month name |
| month_short | STRING | | Abbreviated month (Jan, Feb, ...) |
| week_of_year | BIGINT | | ISO week number |
| day_of_month | BIGINT | | Day of month (1–31) |
| day_of_week | BIGINT | | Day of week (1=Monday, 7=Sunday) |
| day_name | STRING | | Full day name |
| is_weekend | BIGINT | | Weekend flag (0/1) |
| year_month | STRING | | YYYY-MM format |
| year_quarter | STRING | | YYYY-Q# format |
| fiscal_year | BIGINT | | Fiscal year (Oct–Sep cycle) |

---

#### `dp_dim_recommendation` — Action Categories

| Column | Type | Constraint | Description |
| --- | --- | --- | --- |
| recommendation_id | BIGINT | **PK, NOT NULL** | Action category identifier |
| recommendation_name | STRING | | Category name: Continue / Monitor / Improve / Replace |
| description | STRING | | Detailed action description |
| action_priority | BIGINT | | Priority rank (1=highest/Replace, 4=lowest/Continue) |
| color_code | STRING | | Hex color for visualization |

---

#### `dp_dim_carbon_score` — Carbon Score Classifications

| Column | Type | Constraint | Description |
| --- | --- | --- | --- |
| carbon_score_id | STRING | **PK, NOT NULL** | Score grade: A, B, or C |
| score_label | STRING | | Label: Low / Medium / High Carbon |
| description | STRING | | Threshold description |
| intensity_threshold_min | DOUBLE | | Lower bound of carbon intensity range |
| intensity_threshold_max | DOUBLE | | Upper bound of carbon intensity range |

---

### Fact Tables

#### `dp_fact_risk_assessment` — Supplier Risk Scores

| Column | Type | Constraint | Description |
| --- | --- | --- | --- |
| supplier_id | STRING | **PK, NOT NULL, FK→dim_supplier** | Supplier identifier |
| financial_risk_score | DOUBLE | | Financial risk (0–100 scale) |
| sustainability_risk_score | DOUBLE | | Sustainability risk (0–100 scale) |
| operational_risk_score | DOUBLE | | Operational risk (0–100 scale) |
| overall_risk_score | DOUBLE | | Weighted composite risk (Fin 40%, Sust 30%, Ops 30%) |
| high_risk_flag | BIGINT | | Binary risk indicator (0/1) |
| recommendation_id | BIGINT | **FK→dim_recommendation** | Action category assignment |
| carbon_score_id | STRING | **FK→dim_carbon_score** | Carbon grade (A/B/C) |
| bankruptcy_probability_pct | DOUBLE | | D&B bankruptcy probability (%) |
| viability_score | BIGINT | | D&B viability score |
| paydex_score | BIGINT | | D&B payment performance score |
| financial_stress_score | DOUBLE | | D&B financial stress indicator |
| late_payment_pct | DOUBLE | | Percentage of late payments |
| business_continuity_score | DOUBLE | | D&B business continuity score |

---

#### `dp_fact_carbon_emissions` — Scope 3 Emissions

| Column | Type | Constraint | Description |
| --- | --- | --- | --- |
| supplier_id | STRING | **PK, NOT NULL, FK→dim_supplier** | Supplier identifier |
| total_emissions_kg | DOUBLE | | Total CO2 emissions (kg) |
| carbon_intensity | DOUBLE | | Emissions per unit produced (kg CO2/unit) |
| carbon_score_id | STRING | **FK→dim_carbon_score** | Carbon grade (A/B/C) |
| electricity_emissions_kg | DOUBLE | | Electricity CO2 (adjusted for renewables) |
| gas_emissions_kg | DOUBLE | | Natural gas CO2 |
| truck_emissions_kg | DOUBLE | | Truck transport CO2 |
| ship_emissions_kg | DOUBLE | | Ship transport CO2 |
| landfill_emissions_kg | DOUBLE | | Landfill waste CO2 |
| recycling_emissions_kg | DOUBLE | | Recycling process CO2 |
| renewable_energy_pct | BIGINT | | Renewable energy percentage (%) |
| annual_production_units | BIGINT | | Annual production volume |
| electricity_kwh | BIGINT | | Electricity consumption (kWh) |
| natural_gas_kwh | BIGINT | | Natural gas consumption (kWh) |
| truck_distance_km | BIGINT | | Truck transport distance (km) |
| ship_distance_km | BIGINT | | Ship transport distance (km) |
| landfill_waste_kg | BIGINT | | Landfill waste (kg) |
| recycled_waste_kg | BIGINT | | Recycled waste (kg) |

---

#### `dp_fact_supplier_spend` — Procurement Spend

| Column | Type | Constraint | Description |
| --- | --- | --- | --- |
| supplier_id | STRING | **PK, NOT NULL, FK→dim_supplier** | Supplier identifier |
| total_spend_amount | DOUBLE | | Total procurement spend ($) |
| invoice_count | BIGINT | | Number of invoices |
| avg_order_value | DOUBLE | | Average order value ($) |
| unique_items_count | BIGINT | | Number of distinct items purchased |
| late_payment_pct | DOUBLE | | Percentage of overdue payments |
| first_order_date_id | DOUBLE | | First order date (YYYYMMDD, FK→dim_time) |
| last_order_date_id | DOUBLE | | Last order date (YYYYMMDD, FK→dim_time) |
| first_invoice_date_id | DOUBLE | | First invoice date (YYYYMMDD, FK→dim_time) |
| last_invoice_date_id | DOUBLE | | Last invoice date (YYYYMMDD, FK→dim_time) |

---

### Foreign Key Relationships

| Source Table | FK Column | Target Table | PK Column |
| --- | --- | --- | --- |
| dp_fact_risk_assessment | supplier_id | dp_dim_supplier | supplier_id |
| dp_fact_risk_assessment | recommendation_id | dp_dim_recommendation | recommendation_id |
| dp_fact_risk_assessment | carbon_score_id | dp_dim_carbon_score | carbon_score_id |
| dp_fact_carbon_emissions | supplier_id | dp_dim_supplier | supplier_id |
| dp_fact_carbon_emissions | carbon_score_id | dp_dim_carbon_score | carbon_score_id |
| dp_fact_supplier_spend | supplier_id | dp_dim_supplier | supplier_id |
| dp_fact_supplier_spend | first/last_order_date_id | dp_dim_time | date_id |
| dp_fact_supplier_spend | first/last_invoice_date_id | dp_dim_time | date_id |

---

## Conclusion

This **ML-Driven Supplier Risk & Sustainability Intelligence System** delivers:

### Model Performance
- Gradient Boosting Classifier trained on financial, sustainability, and operational features
- Predictive capability to identify at-risk suppliers before disruptions occur
- Feature importance analysis revealing key risk drivers

### Actionable Intelligence
- Every supplier classified into **Continue / Monitor / Improve / Replace** categories
- Quantified financial exposure from high-risk suppliers
- Clear improvement paths for sustainability enhancement

### ESG & Compliance
- Full Scope 3 emissions accounting across the supply base
- Carbon intensity benchmarking and scoring (A/B/C)
- Ready-to-report metrics for ESG disclosure requirements

### Business Value
- **Procurement**: Data-driven supplier selection and retention decisions
- **Risk Management**: Proactive identification of disruption-prone suppliers
- **Sustainability**: Targeted interventions for maximum CO2 reduction impact

## Next Steps
Let us now consume the enriched data product in the SAP BDC Catalog. In the next steps, you will use this data product for modelling in order to consume it in analytical use cases.